To convert a GoogleNet pth file to Core ML to be use in a iPhone app

Install required libraries.

In [1]:
pip install torch torchvision coremltools

Note: you may need to restart the kernel to use updated packages.


Export block
First make it compatible with CoreML

In [5]:
class TrainConfig:
    pass

ckpt = torch.load(
    "best_googlenet_finetuned_realwaste.pth",
    map_location="cpu",
    weights_only=False
)

Inspect it

In [6]:
print(type(ckpt))

if isinstance(ckpt, dict):
    print(ckpt.keys())

<class 'dict'>
dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'best_val_loss', 'config'])


Then save a clean checkpoint:

In [7]:
state = (
    ckpt.get("state_dict")
    or ckpt.get("model_state_dict")
    or ckpt.get("model")
)

torch.save(state, "googlenet_state_dict.pth")

In [ ]:
Export

In [10]:
import torch
import torchvision.models as models
import coremltools as ct

NUM_CLASSES = 9  # change if your checkpoint was fine-tuned

# 1) Recreate GoogLeNet architecture
model = models.googlenet(
    weights=None,
    aux_logits=False,
    num_classes=NUM_CLASSES
)

# 2) Load .pth
### best_googlenet_finetuned_realwaste.pth
state = torch.load("googlenet_state_dict.pth", map_location="cpu")
if "state_dict" in state:
    state = state["state_dict"]

# Optional: remove DataParallel "module." prefix
state = {k.replace("module.", ""): v for k, v in state.items()}

model.load_state_dict(state, strict=True)
model.eval()

# 3) Trace to TorchScript
example_input = torch.rand(1, 3, 224, 224)
traced = torch.jit.trace(model, example_input)

# 4) Convert to Core ML
mlmodel = ct.convert(
    traced,
    inputs=[
        ct.ImageType(
            name="input",
            shape=example_input.shape,
            scale=1 / 255.0,
            bias=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
            color_layout=ct.colorlayout.RGB,
        )
    ],
    classifier_config=ct.ClassifierConfig("labels.txt")  # optional
)

mlmodel.save("GoogLeNet.mlpackage")

When both 'convert_to' and 'minimum_deployment_target' not specified, 'convert_to' is set to "mlprogram" and 'minimum_deployment_target' is set to ct.target.iOS15 (which is same as ct.target.macOS12). Note: the model will not run on systems older than iOS15/macOS12/watchOS8/tvOS15. In order to make your model run on older system, please set the 'minimum_deployment_target' to iOS14/iOS13. Details please see the link: https://apple.github.io/coremltools/docs-guides/source/target-conversion-formats.html
Running MIL backend_mlprogram pipeline: 100%|█████████████████████████████████████| 12/12 [00:00<00:00, 372.84 passes/s]
